# 1. Data Cleaning and Preparation.

# 1. Очистка и подготовка данных.

Import required libraries for cleaning Deals, Spend, Calls, Contacts data.

In [1]:
from pathlib import Path
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
from datetime import time
import re

Uploading unprocessed files, checking the project structure and verify that all required source files are present before processing the data.

Загружаем неочищенные файлы, проверяю структуру проекта и наличие необходимых исходных файлов перед обработкой данных.

In [2]:
CWD = Path.cwd()
PROJECT_DIR = CWD.parent
RAW_DIR = PROJECT_DIR / "raw"
OUT_DIR = PROJECT_DIR / "processed"  
OUT_DIR.mkdir(parents=True, exist_ok=True)

 # Paths
print("CWD:", CWD)
print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_DIR:", RAW_DIR)
print("OUT_DIR:", OUT_DIR)

# --- Fail-fast validation ---
REQUIRED_FILES = ["Contacts.xlsx", "Calls.xlsx", "Spend.xlsx", "Deals.xlsx"]

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Raw folder not found: {RAW_DIR}")

missing = [f for f in REQUIRED_FILES if not (RAW_DIR / f).exists()]
if missing:
    raise FileNotFoundError(" Missing files in raw folder:\n" + "\n".join(missing))

print("Raw folder and files are OK.")


CWD: c:\Users\Dell 7530\Documents\ICH\Python for DA\Final project_Serebrenikova\new_script_clean
PROJECT_DIR: c:\Users\Dell 7530\Documents\ICH\Python for DA\Final project_Serebrenikova
RAW_DIR: c:\Users\Dell 7530\Documents\ICH\Python for DA\Final project_Serebrenikova\raw
OUT_DIR: c:\Users\Dell 7530\Documents\ICH\Python for DA\Final project_Serebrenikova\processed
Raw folder and files are OK.


Helper Functions.
Вспомогательные функции.

In [3]:

def drop_all_null_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns where all values are missing."""
    return df.dropna(axis=1, how="all")


def drop_duplicate_rows(df: pd.DataFrame, subset: Optional[List[str]] = None) -> pd.DataFrame:
    """Drop duplicate rows (optionally by key columns)."""
    return df.drop_duplicates(subset=subset, keep="first")


def cast_to_string(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Cast ONLY selected columns to pandas 'string' dtype."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "<NA>": pd.NA})
            )
    return df


def cast_to_datetime(df: pd.DataFrame, cols: List[str], dayfirst: bool = True) -> pd.DataFrame:
    """Parse ONLY selected columns as datetime (invalid -> NaT (Not a Time))."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=dayfirst)
    return df

# safe Excel ID/string converter (ONLY for selected columns)
def excel_to_string_safe(x: object) -> object:
    """Force Excel-read values to clean string. Prevents scientific notation (e+18)."""
    if pd.isna(x):
        return pd.NA
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        if float(x).is_integer():
            return str(int(x))
        return str(x)
    s = str(x).strip()
    return pd.NA if s == "" else s


def clean_euro(cell: object) -> float:
    """Convert European-formatted money strings into float.
    - Remove '€'
    - Remove thousand separators '.'
    - Replace ',' -> '.'"""
    if isinstance(cell, str):
        cell = cell.replace("€", "")
        cell = cell.replace(".", "")
        cell = cell.replace(",", ".")
        cell = cell.strip()
        try:
            return float(cell)
        except ValueError:
            return float("nan")
    elif pd.isna(cell):
        return float("nan")
    else:
        try:
            return float(cell)
        except ValueError:
            return float("nan")
        
        
def status_sla(sla):
    """Categorizes SLA duration into discrete time buckets for analysis.

    Converts the input timedelta object into minutes and assigns a categorical 
    label based on predefined intervals (e.g., '0-5 min', '1d - 2d'). 
    Handles null or missing values by returning 'Unknown'.

    Parameters:
    sla (pd.Timedelta or NaN): The SLA duration value to categorize.

    Returns:
    str: A string representing the assigned time bucket category."""
    if pd.isna(sla):
        return 'Unknown'

    minutes = sla.total_seconds() / 60

    if 0 <= minutes < 6:
        return '0-5 min'
    elif 6 <= minutes < 11:
        return '5-10 min'
    elif 11 <= minutes < 31:
        return '10-30 min'
    elif 31 <= minutes < 61:
        return '30 min - 1h'
    elif 61 <= minutes < 91:
        return '1 - 1.5h'
    elif 91 <= minutes < 181:
        return '1.5 - 3h'
    elif 181 <= minutes < 361:
        return '3 - 6h'
    elif 361 <= minutes < 721:
        return '6 - 12h'
    elif 721 <= minutes < 1441:
        return '12 - 24h'
    elif 1441 <= minutes < 2881:
        return '1d - 2d'
    elif 2881 <= minutes < 10081:
        return '2d - 7d'
    elif 10081 <= minutes < 40321:
        return '7d - 1M'
    elif 40321 <= minutes < 241921:
        return '1M - 6M'
    elif 241921 <= minutes:
        return '6M+'

def print_snapshot(df: pd.DataFrame, label: str) -> None: 
    """the general purpose of the function: EDA (Exploratory Data Analysis):
    Print a compact snapshot of a dataset.
    Docstring, header output, size output, datatype output, gap analysis, top-gap output
    """                                                                             #
    print(f"\n=== {label} ===")                                                    
    print("Shape:", df.shape)                                                       
    print("\nDtypes:\n", df.dtypes.astype(str)) 
    #
    print(f"\n🔧 Dates Swapped (Closing < Created): {df.attrs.get('dates_swapped', 0)}")                                  
    miss = df.isna().sum().sort_values(ascending=False)                             
    print(f"\n missing:\n", miss) 
    

Load Raw Data.
Загрузка сырых данных.

In [4]:
# --- Contacts ---
df_contacts_raw = pd.read_excel(
    RAW_DIR / "Contacts.xlsx",
    converters={
        "Id": excel_to_string_safe,
        "Contact Owner Name": excel_to_string_safe,
        "CONTACTID": excel_to_string_safe,
    },
)
df_contacts_raw = cast_to_datetime(
    df_contacts_raw,
    ["Created Time", "Modified Time"]
)

# --- Calls ---
df_calls_raw = pd.read_excel(
    RAW_DIR / "Calls.xlsx",
    converters={
        "Id": excel_to_string_safe,
        "Contact Owner Name": excel_to_string_safe,
        "CONTACTID": excel_to_string_safe,
    },
)
df_calls_raw = cast_to_datetime(
    df_calls_raw,
    ["Created Time", "Modified Time"]
)

# --- Deals -
df_deals_raw = pd.read_excel(
    RAW_DIR / "Deals.xlsx",
    converters={
        "Id": excel_to_string_safe,
        "Contact Owner Name": excel_to_string_safe,
        "CONTACTID": excel_to_string_safe,
        "Contact Name": excel_to_string_safe,
    },
)
df_deals_raw = cast_to_datetime(
    df_deals_raw,
    ["Created Time", "Modified Time"]
)

# Spend 
df_spend_raw = pd.read_excel(RAW_DIR / "Spend.xlsx")
df_spend_raw = cast_to_datetime(
    df_spend_raw,
    ["Date"]
)

# print before cleaning
print_snapshot(df_contacts_raw, "Contacts_RAW")
print_snapshot(df_calls_raw, "Calls_RAW")
print_snapshot(df_spend_raw, "Spend_RAW")
print_snapshot(df_deals_raw, "Deals_RAW")


=== Contacts_RAW ===
Shape: (18548, 4)

Dtypes:
 Id                            object
Contact Owner Name            object
Created Time          datetime64[ns]
Modified Time         datetime64[ns]
dtype: object

🔧 Dates Swapped (Closing < Created): 0

 missing:
 Id                    0
Contact Owner Name    0
Created Time          0
Modified Time         0
dtype: int64

=== Calls_RAW ===
Shape: (95874, 11)

Dtypes:
 Id                             object
Call Start Time                object
Call Owner Name                object
CONTACTID                      object
Call Type                      object
Call Duration (in seconds)    float64
Call Status                    object
Dialled Number                float64
Outgoing Call Status           object
Scheduled in CRM              float64
Tag                           float64
dtype: object

🔧 Dates Swapped (Closing < Created): 0

 missing:
 Tag                           95874
Dialled Number                95874
Scheduled in CRM       

Cleaning function Contacts. 

In [5]:
def clean_contacts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Contacts cleaning:
    - ONLY these columns to string: 'Id', 'Contact Owner Name', 'CONTACTID' (if exists)
    - Parse dates: 'Created Time', 'Modified Time'
    - Drop duplicates by 'Id'
    """
    df = drop_all_null_columns(df)
    df = cast_to_string(df, ["Id", "Contact Owner Name", "CONTACTID"])
    df = cast_to_datetime(df, ["Created Time", "Modified Time"], dayfirst=True)
    df = drop_duplicate_rows(df, subset=["Id"] if "Id" in df.columns else None)
    return df

df_contacts = clean_contacts(df_contacts_raw)

print_snapshot(df_contacts, "Contacts_CLEAN")



=== Contacts_CLEAN ===
Shape: (18548, 4)

Dtypes:
 Id                            string
Contact Owner Name            string
Created Time          datetime64[ns]
Modified Time         datetime64[ns]
dtype: object

🔧 Dates Swapped (Closing < Created): 0

 missing:
 Id                    0
Contact Owner Name    0
Created Time          0
Modified Time         0
dtype: int64


Cleaning function: Calls

In [6]:
def clean_calls(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calls cleaning:
    - Drop 'Dialled Number' and 'Tag'
    - Cast ONLY: 'Id', 'CONTACTID' to string (CONTACTID must not be float)
    - Parse 'Call Start Time' to datetime (if possible)
    - Drop duplicates
    """
    df = drop_all_null_columns(df)

    for col in ["Dialled Number", "Tag"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    df = cast_to_string(df, ["Id", "CONTACTID"])
    df = cast_to_datetime(df, ["Call Start Time"], dayfirst=True)

    df = drop_duplicate_rows(df)
    return df

df_calls = clean_calls(df_calls_raw)

print_snapshot(df_calls, "Calls_CLEAN")

print("\nCONTACTID dtype after cleaning:", df_calls["CONTACTID"].dtype)



=== Calls_CLEAN ===
Shape: (95874, 9)

Dtypes:
 Id                                    string
Call Start Time               datetime64[ns]
Call Owner Name                       object
CONTACTID                             string
Call Type                             object
Call Duration (in seconds)           float64
Call Status                           object
Outgoing Call Status                  object
Scheduled in CRM                     float64
dtype: object

🔧 Dates Swapped (Closing < Created): 0

 missing:
 Outgoing Call Status          8999
Scheduled in CRM              8999
CONTACTID                     3933
Call Duration (in seconds)      83
Id                               0
Call Type                        0
Call Owner Name                  0
Call Start Time                  0
Call Status                      0
dtype: int64

CONTACTID dtype after cleaning: string


Cleaning function: Spend

In [7]:
def clean_spend(df: pd.DataFrame) -> pd.DataFrame:
    """
    Spend cleaning:
    - Parse 'Date' to datetime
    - Convert Impressions/Spend/Clicks to numeric
    - Remove rows where Impressions=0 AND Spend=0 AND Clicks=0
    - Drop duplicates
    """
    df = drop_all_null_columns(df)
    df = cast_to_datetime(df, ["Date"], dayfirst=True)

    for col in ["Impressions", "Spend", "Clicks"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = drop_duplicate_rows(df)

    df['Campaign'] = df['Campaign'].fillna('Unknown')
    
    return df

df_spend = clean_spend(df_spend_raw)

print_snapshot(df_spend, "Spend_CLEAN")


=== Spend_CLEAN ===
Shape: (19862, 8)

Dtypes:
 Date           datetime64[ns]
Source                 object
Campaign               object
Impressions             int64
Spend                 float64
Clicks                  int64
AdGroup                object
Ad                     object
dtype: object

🔧 Dates Swapped (Closing < Created): 0

 missing:
 AdGroup        5911
Ad             5911
Date              0
Source            0
Impressions       0
Campaign          0
Clicks            0
Spend             0
dtype: int64


Cleaning function: Deals 

In [8]:
def avg_duration(df):
# 4. Fill missing closing date for closed deals 
    # Filter closed deals with completed dates  
    avg_closed_operations = (
        df["Stage"].isin(["Lost", "Payment Done"]) &
        df["Created Time"].notna() &
        df["Closing Date"].notna()
    )
    # Calculate Average duration in days
    avg_duration = (
        df.loc[avg_closed_operations, "Closing Date"] -
        df.loc[avg_closed_operations, "Created Time"]
    ).dt.total_seconds().mean() / (3600 * 24)

    # Safety check for empty Result
    if pd.isna(avg_duration):
        avg_duration = 0

    avg_duration_days = round(avg_duration)
    print("Deals: average closed duration (days, rounded):", avg_duration_days)

    # Find Deals missing closing date
    closed_no_date = (
        df["Stage"].isin(["Lost", "Payment Done"]) &
        df["Closing Date"].isna() &
        df["Created Time"].notna()
    )
    # Count and print missing dates
    print("Deals: closed deals without Closing Date:", int(closed_no_date.sum()))

    # Fill missing dates with calculated value
    df.loc[closed_no_date, "Closing Date"] = (
        df.loc[closed_no_date, "Created Time"] + pd.Timedelta(days=avg_duration)
    )
    # Remove the time component from Dates
    df["Closing Date"] = df["Closing Date"].dt.normalize()
    return df

def numeric_columns(df):
# 5. Convert numeric columns (Months of study, Course duration)
    # Converts to numeric, forcing errors to NaN to prevent crashes
    for col in ["Months of study", "Course duration"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # 6 Cleaning money columns
    for col in ["Initial Amount Paid", "Offer Total Amount"]:
        if col in df.columns:
            # Step 1: Clean euro format (remove symbols, fix decimals)
            df[col] = df[col].apply(clean_euro).astype("float64")
            # Step 2: Safe numeric conversion (errors -> NaN)
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
            
     # 6.5 Fix Logic: Swap if Initial Amount > Offer Total Amount
    if "Initial Amount Paid" in df.columns and "Offer Total Amount" in df.columns:
        mask_swap = df["Initial Amount Paid"] > df["Offer Total Amount"]
        count_swap = int(mask_swap.sum())
        
        if count_swap > 0:
            df.loc[mask_swap, ["Initial Amount Paid", "Offer Total Amount"]] = \
                df.loc[mask_swap, ["Offer Total Amount", "Initial Amount Paid"]].values
            print(f"Deals: swapped values where Initial > Offer Total:", count_swap)

    return df

def fix_date_logic(df):
    # Fix Logic: Swap if Closing Date < Created Time
    if "Created Time" in df.columns and "Closing Date" in df.columns:
        mask_swap = df["Closing Date"] < df["Created Time"]
        count_swap = int(mask_swap.sum())
        
        if count_swap > 0:
            df.loc[mask_swap, ["Created Time", "Closing Date"]] = \
                df.loc[mask_swap, ["Closing Date", "Created Time"]].values
            print(f"Deals: swapped dates where Closing < Created:", count_swap)
        else:
            print("Deals: all dates are logically correct (Closing >= Created)")
            df.attrs["dates_swapped"] = count_swap
    else:
        print("Deals: all dates are logically correct (Closing >= Created)")
        df.attrs["dates_swapped"] = 0   
    return df

def normalize_level(val):
    'Приводим записи к формату A/B/C0-2, включая кириллицу.'
    if pd.isna(val):
        return pd.NA
    text = str(val).strip()
    text = re.sub(r"[Аа]", "A", text)  # Аа -> A
    text = re.sub(r"[ВвБб]", "B", text)  # ВвБб -> B
    text = re.sub(r"[Сс]", "C", text)  # Сс -> C
    text = text.upper().replace(" ", "")
    match = re.search(r"[ABC][0-2]", text)
    return match.group() if match else pd.NA

In [9]:
def clean_deals(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deals cleaning (fixed, keeps row count correct):
    - Replace Excel error '#REF!' and empty-like strings with NA
    - Drop fully empty rows 
    - Drop column 'Page'
    - Cast selected ID/name columns to string
    - Parse dates: 'Created Time', 'Closing Date'
    - Fill missing Closing Date for closed stages using average duration
    - Convert 'Months of study' and 'Course duration' to numeric
    - Clean money columns: 'Initial Amount Paid', 'Offer Total Amount'
    - Replace 0 in Initial Amount Paid with NaN
    - Fill missing 'Deal Owner Name' with 'Unknown'
    - If first payment is NaN -> set 'Payment Done' = 'Unknown stage'
    - SLA LAST: convert SLA to minutes + SLA_status categories
    """
    df = df.copy()


    # 1 Drop
    df = df.replace("#REF!", pd.NA)                  # Replace Excel error '#REF!
    df = df.replace("\u00A0", pd.NA)                 # non-breaking space
    df = df.replace(r"^\s*$", pd.NA, regex=True)     # whitespace-only -> NA

    df = drop_all_null_columns(df)  # Drop fully-null columns 

    # Drop 'Page' column (requested)
    if "Page" in df.columns:
        df = df.drop(columns=["Page"])

    # Drop fully empty rows  
    before_rows = df.shape[0]
    df = df.dropna(how="all")
    print("Deals: dropped fully empty rows (after #REF!/whitespace cleanup):", before_rows - df.shape[0])

    # 2 Cast selected ID/name columns to string 
    df = cast_to_string(df, ["Id", "Deal Owner Name", "Contact Name", "CONTACTID", "Contact Owner Name"])

    # 3 Parse dates
    df = cast_to_datetime(df, ["Created Time", "Closing Date"], dayfirst=True)
    
    # 3.1 Fix date logic (swap if Closing < Created)
    df = fix_date_logic(df)

    df = avg_duration(df)

    df = numeric_columns(df)
    
    # 8 Cast selected Level of Deutsch columns to string
    df = cast_to_string(df, ["Level of Deutsch"])

    # 9 Fillna Deal Owner Name
    if "Deal Owner Name" in df.columns:
        df["Deal Owner Name"] = df["Deal Owner Name"].fillna("Unknown")
    
    # 10  SLA in minutes
    df['SLA'] = (df['SLA'].astype(str).replace("nan", None))
    df['SLA'] = pd.to_timedelta(df['SLA'], errors='coerce')
    df['SLA_status'] = df['SLA'].apply(lambda sla: status_sla(sla))

    df["level_norm"] = df["Level of Deutsch"].map(normalize_level)

    df['Campaign'] = df['Campaign'].fillna('Unknown')

    return df

# IMPORTANT: Do NOT drop duplicates by Id. I tried this before and ended up with only 8613 rows.    

Validation Deals functions 

In [10]:
df_deals = clean_deals(df_deals_raw)
print_snapshot(df_deals, "Deals_clean_FINAL")
print("FINAL SHAPE:", df_deals.shape)

Deals: dropped fully empty rows (after #REF!/whitespace cleanup): 2
Deals: swapped dates where Closing < Created: 3312
Deals: average closed duration (days, rounded): 15
Deals: closed deals without Closing Date: 2222
Deals: swapped values where Initial > Offer Total: 58

=== Deals_clean_FINAL ===
Shape: (21593, 24)

Dtypes:
 Id                              string
Deal Owner Name                 string
Closing Date            datetime64[ns]
Quality                         object
Stage                           object
Lost Reason                     object
Campaign                        object
SLA                    timedelta64[ns]
Content                         object
Term                            object
Source                          object
Payment Type                    object
Product                         object
Education Type                  object
Created Time            datetime64[ns]
Course duration                float64
Months of study                float64
Initial Am

Check SLA distribution by group (before saving).
Проверка распределения SLA по группам (перед сохранением).

In [11]:
print("\n=== SLA Distribution by Status Groups ===\n")

# Group by SLA_status and count deals
sla_summary = df_deals.groupby("SLA_status", dropna=False).size().reset_index(name="Count")

# Calculate percentage of total count
total = sla_summary["Count"].sum()
sla_summary["Percentage"] = round((sla_summary["Count"] / total) * 100, 2)

# Define sort order by time duration (shortest to longest)
sla_order = [
    'Unknown', '0-5 min', '5-10 min', '10-30 min', '30 min - 1h',
    '1 - 1.5h', '1.5 - 3h', '3 - 6h', '6 - 12h', '12 - 24h',
    '1d - 2d', '2d - 7d', '7d - 1M', '1M - 6M', '6M+'
]

sla_summary["SLA_status"] = pd.Categorical(
    sla_summary["SLA_status"], 
    categories=sla_order, 
    ordered=True
)
sla_summary = sla_summary.sort_values("SLA_status")

print(sla_summary.to_string(index=False))
print(f"\nTotal deals: {total}")


=== SLA Distribution by Status Groups ===

 SLA_status  Count  Percentage
    Unknown   6060       28.06
    0-5 min    291        1.35
   5-10 min    356        1.65
  10-30 min   1543        7.15
30 min - 1h   1317        6.10
   1 - 1.5h    886        4.10
   1.5 - 3h   1675        7.76
     3 - 6h   1929        8.93
    6 - 12h   1682        7.79
   12 - 24h   3995       18.50
    1d - 2d    741        3.43
    2d - 7d    696        3.22
    7d - 1M    314        1.45
    1M - 6M     94        0.44
        6M+     14        0.06

Total deals: 21593


Verification: Check data types and NaN counts for numeric columns

In [12]:
df_deals = clean_deals(df_deals_raw)

# Money columns
print("\n=== Money Columns ===")
print("Initial Amount Paid dtype:", df_deals["Initial Amount Paid"].dtype)
print("Offer Total Amount dtype:", df_deals["Offer Total Amount"].dtype)
print("Initial Amount Paid NaN count:", df_deals["Initial Amount Paid"].isna().sum())
print("Offer Total Amount NaN count:", df_deals["Offer Total Amount"].isna().sum())

# Numeric columns (Months of study, Course duration)
print("\n=== Numeric Columns ===")
print("Months of study dtype:", df_deals["Months of study"].dtype)
print("Course duration dtype:", df_deals["Course duration"].dtype)
print("Months of study NaN count:", df_deals["Months of study"].isna().sum())
print("Course duration NaN count:", df_deals["Course duration"].isna().sum())

Deals: dropped fully empty rows (after #REF!/whitespace cleanup): 2
Deals: swapped dates where Closing < Created: 3312
Deals: average closed duration (days, rounded): 15
Deals: closed deals without Closing Date: 2222
Deals: swapped values where Initial > Offer Total: 58

=== Money Columns ===
Initial Amount Paid dtype: float64
Offer Total Amount dtype: float64
Initial Amount Paid NaN count: 17428
Offer Total Amount NaN count: 17408

=== Numeric Columns ===
Months of study dtype: float64
Course duration dtype: float64
Months of study NaN count: 20753
Course duration NaN count: 18006


Saving final files and outputting directory path.
Сохранение финальных файлов, вывод пути к директории.

In [13]:
outputs = {
    "Contacts": df_contacts,
    "Calls": df_calls,
    "Spend": df_spend,
    "Deals": df_deals
}

for name, df in outputs.items():
    # Save Excel
    excel_path = OUT_DIR / f"{name}_clean.xlsx"
    df.to_excel(excel_path, index=False)

    # Save Pickle
    pkl_path = OUT_DIR / f"{name}_clean.pkl"
    df.to_pickle(pkl_path)

    # Save Parquet from parquet-safe copy
    parquet_path = OUT_DIR / f"{name}_clean.parquet"
    df.to_parquet(parquet_path, index=False)

    print(f"\n Saved {name}:")
    print(" -", excel_path.name)
    print(" -", pkl_path.name)
    print(" -", parquet_path.name)

print("\nFINAL OUTPUT FILES IN:", OUT_DIR)
print([p.name for p in OUT_DIR.glob("*_clean.*")])



 Saved Contacts:
 - Contacts_clean.xlsx
 - Contacts_clean.pkl
 - Contacts_clean.parquet

 Saved Calls:
 - Calls_clean.xlsx
 - Calls_clean.pkl
 - Calls_clean.parquet

 Saved Spend:
 - Spend_clean.xlsx
 - Spend_clean.pkl
 - Spend_clean.parquet

 Saved Deals:
 - Deals_clean.xlsx
 - Deals_clean.pkl
 - Deals_clean.parquet

FINAL OUTPUT FILES IN: c:\Users\Dell 7530\Documents\ICH\Python for DA\Final project_Serebrenikova\processed
['Calls_clean.parquet', 'Calls_clean.pkl', 'Calls_clean.xlsx', 'Contacts_clean.parquet', 'Contacts_clean.pkl', 'Contacts_clean.xlsx', 'Deals_clean.parquet', 'Deals_clean.pkl', 'Deals_clean.xlsx', 'Spend_clean.parquet', 'Spend_clean.pkl', 'Spend_clean.xlsx']
